<div style="padding: 20px; background: linear-gradient(90deg, #155e75 0%, #7c3aed 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">Module 5.5: Ensemble Retriever</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Blend sparse and dense retrievers with weighted rank fusion.</p>
</div>

---

### Course alignment and free-first stack

- Covers: Ensemble retrieval after similarity search, score thresholds, MMR, and hybrid search.
- Runtime stack: BM25, Chroma, and local Hugging Face sentence-transformers embeddings. No paid API key is required.
- Current LangChain pattern: retrievers are invoked with `.invoke()`, then fused in transparent Python so the ranking math is visible.


## 1. Why an ensemble retriever?

Different retrievers make different mistakes:

- Dense vector search is strong for semantic paraphrases.
- BM25 is strong for exact terms, IDs, acronyms, and rare keywords.
- MMR can improve diversity but may miss exact lexical matches.

An ensemble retriever runs multiple retrievers and combines their rankings. The usual production pattern is weighted reciprocal rank fusion: high-ranked results from any reliable retriever get promoted, while duplicates are merged.


In [ ]:
import os
from collections import defaultdict
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

os.environ["TOKENIZERS_PARALLELISM"] = "false"

embedding_model = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

docs = [
    Document(page_content="Model XQ-9000 overheats when the filter is clogged.", metadata={"source": "manual"}),
    Document(page_content="The blender motor can become hot if airflow is blocked.", metadata={"source": "support"}),
    Document(page_content="Model XQ-8000 uses a smaller motor and a different filter.", metadata={"source": "manual"}),
    Document(page_content="Reset the device after cleaning the intake vents.", metadata={"source": "support"}),
    Document(page_content="RAG systems combine retrieval and generation for grounded answers.", metadata={"source": "course"}),
]

bm25 = BM25Retriever.from_documents(docs)
bm25.k = 3

vectorstore = Chroma.from_documents(docs, embeddings, collection_name="ensemble_demo")
dense = vectorstore.as_retriever(search_kwargs={"k": 3})


## 2. Weighted reciprocal rank fusion

For each retriever result, assign a score based on rank:

`weight / (rank_constant + rank)`

A document gets contributions from every retriever that found it. Larger weights favor one retriever over another. Larger rank constants reduce the gap between rank 1 and rank 2.


In [ ]:
def doc_key(doc: Document) -> str:
    return doc.page_content


def weighted_rrf(results_by_retriever, weights, rank_constant: int = 60, top_k: int = 4):
    scores = defaultdict(float)
    docs_by_key = {}

    for retriever_name, docs_for_retriever in results_by_retriever.items():
        weight = weights.get(retriever_name, 1.0)
        for rank, doc in enumerate(docs_for_retriever, start=1):
            key = doc_key(doc)
            docs_by_key[key] = doc
            scores[key] += weight / (rank_constant + rank)

    ranked = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    return [(docs_by_key[key], score) for key, score in ranked[:top_k]]


def ensemble_retrieve(query: str, top_k: int = 4):
    sparse_docs = bm25.invoke(query)
    dense_docs = dense.invoke(query)

    fused = weighted_rrf(
        {"bm25": sparse_docs, "dense": dense_docs},
        weights={"bm25": 0.55, "dense": 0.45},
        top_k=top_k,
    )
    return sparse_docs, dense_docs, fused


## 3. Compare sparse, dense, and ensemble results

The query mixes an exact product ID with a semantic symptom. A good retriever should preserve the ID match while also considering overheating context.


In [ ]:
query = "Why does Model XQ-9000 get hot?"
sparse_docs, dense_docs, fused_docs = ensemble_retrieve(query)

print("BM25 results")
for doc in sparse_docs:
    print(f"- {doc.page_content}")

print("\nDense results")
for doc in dense_docs:
    print(f"- {doc.page_content}")

print("\nEnsemble results")
for doc, score in fused_docs:
    print(f"- score={score:.4f} | {doc.page_content}")


## 4. When to use ensembles

Use an ensemble when the user queries contain both meaning and exact identifiers: product IDs, legal citations, API names, error codes, patient-safe medical terms, or internal acronyms.

Start with equal weights, inspect failure cases, and tune weights with an evaluation set. Do not tune only by intuition; retrieval changes should be measured with context precision, context recall, and downstream answer quality.
